In [ ]:
!git clone https://github.com/swz30/Restormer.git
%cd Restormer

In [ ]:
!pip install -q einops addict future lmdb numpy pyyaml requests scipy tb-nightly yapf lpips

In [ ]:
!pip install -q matplotlib scikit-learn scikit-image opencv-python yacs joblib natsort h5py tqdm

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
!find /kaggle/input -name "motion_deblurring.pth"

In [ ]:
#11
!cp /kaggle/input/datasets/baohuynhlongthien/motion-deblurring-pretrained/motion_deblurring.pth \
   Motion_Deblurring/pretrained_models/motion_deblurring.pth

In [ ]:
#15
!grep -n "class Restormer\|class TransformerBlock\|class MDTA\|class GDFN" \
    basicsr/models/archs/restormer_arch.py

In [ ]:
!sed -n '1,260p' basicsr/models/archs/restormer_arch.py

In [ ]:
!sed -n '246,330p' basicsr/models/archs/restormer_arch.py

In [ ]:
#18
!cat Motion_Deblurring/Datasets/README.md

In [ ]:
!cat Motion_Deblurring/Options/Deblurring_Restormer.yml

In [ ]:
!sed -n '1,300p' Motion_Deblurring/test.py

In [ ]:
#21
!sed -n '1,260p' Motion_Deblurring/download_data.py

In [ ]:
#24
!pip install -q -U gdown

In [ ]:
#26
!mkdir -p Datasets

In [ ]:

!gdown --continue \
  "https://drive.google.com/uc?id=1k6DTSHu4saUgrGTYkkZXTptILyG9RRll" \
  -O /kaggle/working/Restormer/Datasets/test.zip

In [ ]:
!ls -lh Datasets/test.zip

In [ ]:
!unzip -q Datasets/test.zip -d Datasets

In [ ]:
#30
!rm Datasets/test.zip

In [ ]:
%cd Motion_Deblurring

In [ ]:
!PYTHONPATH=.. python test.py \
    --input_dir ../Datasets/ \
    --result_dir ../results/ \
    --weights ./pretrained_models/motion_deblurring.pth \
    --dataset GoPro

In [ ]:
!find ../results/GoPro -type f | head

In [ ]:
!find ../results/GoPro -type f | wc -l

In [ ]:
import os
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim

gt_dir = "../Datasets/test/GoPro/target"
pred_dir = "../results/GoPro"

gt_files = sorted([
    f for f in os.listdir(gt_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

pred_files = sorted([
    f for f in os.listdir(pred_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

print("GT:", len(gt_files))
print("Prediction:", len(pred_files))

psnr_values = []
ssim_values = []

for filename in gt_files:
    gt_path = os.path.join(gt_dir, filename)
    pred_path = os.path.join(pred_dir, filename)

    gt = cv2.imread(gt_path)
    pred = cv2.imread(pred_path)

    if gt is None or pred is None:
        print("Cannot read:", filename)
        continue

    if gt.shape != pred.shape:
        print("Shape mismatch:", filename, gt.shape, pred.shape)
        continue

    mse = np.mean(
        (gt.astype(np.float64) - pred.astype(np.float64)) ** 2
    )

    if mse == 0:
        psnr = float("inf")
    else:
        psnr = 10 * np.log10((255.0 ** 2) / mse)

    ssim_value = ssim(
        gt,
        pred,
        channel_axis=2,
        data_range=255
    )

    psnr_values.append(psnr)
    ssim_values.append(ssim_value)

print("\n===== Restormer GoPro Test =====")
print(f"Images evaluated : {len(psnr_values)}")
print(f"Average PSNR     : {np.mean(psnr_values):.4f} dB")
print(f"Average SSIM     : {np.mean(ssim_values):.4f}")

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

base = Path("/kaggle/working/Restormer")

input_dir = base / "Datasets/test/GoPro/input"
gt_dir = base / "Datasets/test/GoPro/target"
pred_dir = base / "results/GoPro"

# Chọn một ảnh có tên tồn tại trong cả ba thư mục
common = sorted(
    p.name for p in input_dir.glob("*")
    if p.is_file()
    and (gt_dir / p.name).is_file()
    and (pred_dir / p.name).is_file()
)

print("Số ảnh có đủ cả ba phiên bản:", len(common))

if common:
    name = common[0]

    images = [
        ("Input — ảnh nhòe", Image.open(input_dir / name)),
        ("Restormer — dự đoán", Image.open(pred_dir / name)),
        ("Ground truth — ảnh chuẩn", Image.open(gt_dir / name)),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, (title, img) in zip(axes, images):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Chưa đủ ảnh trong ba thư mục để so sánh.")